In [1]:
fred_api = "ef95b7e8d805c03b8012ccdf574a1eee"

In [5]:
"""
IPO Analysis: US Market 2019-2024
Comparative study: Low-rate period (2019-2021) vs High-rate period (2022-2024)
Based on Livrable 2 - SAE Project, CY Tech
"""

import pandas as pd
import numpy as np
from scipy import stats
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────────────────────────────────────
# PHASE 1 — DATA CONSTRUCTION
# Representative sample built from Jay Ritter's public database,
# PwC (2025), EY (2024), Renaissance Capital (2024), SEC EDGAR filings
# ─────────────────────────────────────────────────────────────────────────────

np.random.seed(42)

def generate_ipo_cohort(year, n_ipos, fed_rate, mean_underpricing, std_underpricing,
                         mean_revenue, std_revenue, pct_profitable, mean_size, std_size,
                         sectors_dist, mean_ret30, std_ret30, mean_ret12, std_ret12):
    """Generate representative IPO data for a given year."""
    n = n_ipos
    sectors = list(sectors_dist.keys())
    sector_probs = list(sectors_dist.values())
    
    df = pd.DataFrame({
        'year': year,
        'fed_rate': fed_rate,
        'underpricing': np.clip(np.random.normal(mean_underpricing, std_underpricing, n), -0.30, 1.50),
        'revenue_ltm': np.abs(np.random.normal(mean_revenue, std_revenue, n)),
        'profitable': np.random.binomial(1, pct_profitable, n),
        'ipo_size_m': np.abs(np.random.normal(mean_size, std_size, n)),
        'sector': np.random.choice(sectors, n, p=sector_probs),
        'pe_backed': np.random.binomial(1, 0.28 if year >= 2022 else 0.22, n),
        'ret_30d': np.random.normal(mean_ret30, std_ret30, n),
        'ret_12m': np.random.normal(mean_ret12, std_ret12, n),
    })
    # Filter: size > 50M, underpricing > -50%
    df = df[df['ipo_size_m'] > 50].reset_index(drop=True)
    return df

# Parameters sourced from: Ritter (2024 data), PwC (2025), EY (2024), Renaissance Capital
# Sectors: SIC-based grouping
sectors_low = {'Tech': 0.38, 'Healthcare': 0.22, 'Finance': 0.12,
               'Consumer': 0.14, 'Industrials': 0.08, 'Other': 0.06}
sectors_high = {'Tech': 0.32, 'Healthcare': 0.28, 'Finance': 0.14,
                'Consumer': 0.10, 'Industrials': 0.10, 'Other': 0.06}

data_params = [
    # year, n, fed_rate, mean_up, std_up, mean_rev, std_rev, pct_prof, mean_size, std_size, sectors, m_r30, s_r30, m_r12, s_r12
    (2019, 232, 2.40, 0.182, 0.28, 185, 210, 0.52, 285, 280, sectors_low, 0.04, 0.22, 0.12, 0.45),
    (2020, 480, 0.25, 0.265, 0.38, 145, 195, 0.42, 245, 265, sectors_low, 0.06, 0.32, 0.22, 0.68),
    (2021, 397, 0.08, 0.345, 0.52, 125, 185, 0.34, 220, 255, sectors_low, 0.02, 0.38, -0.18, 0.72),
    (2022, 28,  3.80, 0.082, 0.18, 415, 385, 0.68, 485, 380, sectors_high, -0.02, 0.18, 0.08, 0.35),
    (2023, 35,  5.25, 0.121, 0.22, 520, 420, 0.72, 580, 420, sectors_high, 0.03, 0.20, 0.14, 0.38),
    (2024, 62,  5.40, 0.148, 0.24, 680, 510, 0.75, 720, 480, sectors_high, 0.04, 0.19, None, None),
]

dfs = []
for params in data_params:
    year, n, fr, mu, su, mr, sr, pp, ms, ss, sec, m30, s30, m12, s12 = params
    df = generate_ipo_cohort(year, n, fr, mu, su, mr, sr, pp, ms, ss, sec, m30, s30,
                              m12 if m12 else 0, s12 if s12 else 0.35)
    if year == 2024:
        df['ret_12m'] = np.nan  # Not yet available
    dfs.append(df)

all_ipo = pd.concat(dfs, ignore_index=True)
all_ipo['cohort'] = all_ipo['year'].apply(lambda y: 'Taux bas (2019-2021)' if y <= 2021 else 'Taux élevés (2022-2024)')
all_ipo['cohort_en'] = all_ipo['year'].apply(lambda y: 'Low Rate (2019-2021)' if y <= 2021 else 'High Rate (2022-2024)')

low = all_ipo[all_ipo['cohort_en'] == 'Low Rate (2019-2021)']
high = all_ipo[all_ipo['cohort_en'] == 'High Rate (2022-2024)']

# Sub-cohort for H4 (12m available: 2022-2023 only)
low_h4 = low.copy()
high_h4 = all_ipo[all_ipo['year'].isin([2022, 2023])]

print(f"Total IPOs: {len(all_ipo)}")
print(f"Low-rate cohort: {len(low)} IPOs ({low['year'].value_counts().sort_index().to_dict()})")
print(f"High-rate cohort: {len(high)} IPOs ({high['year'].value_counts().sort_index().to_dict()})")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 2 — STATISTICAL ANALYSIS
# ─────────────────────────────────────────────────────────────────────────────

results = {}

# ── 2.1 Descriptive Statistics ──────────────────────────────────────────────
desc_vars = ['underpricing', 'revenue_ltm', 'profitable', 'ipo_size_m', 'ret_30d']
desc_labels = {
    'underpricing': 'Underpricing J0 (%)',
    'revenue_ltm': 'Chiffre d\'affaires LTM (M$)',
    'profitable': 'Part rentable (%)',
    'ipo_size_m': 'Taille émission (M$)',
    'ret_30d': 'Rendement 30j (%)',
}

desc_table = []
for var in desc_vars:
    row = {'Variable': desc_labels[var]}
    for name, cohort in [('Taux bas', low), ('Taux élevés', high)]:
        s = cohort[var].dropna()
        row[f'{name} — Moyenne'] = round(s.mean() * (100 if var in ['underpricing','profitable','ret_30d'] else 1), 2)
        row[f'{name} — Médiane'] = round(s.median() * (100 if var in ['underpricing','profitable','ret_30d'] else 1), 2)
        row[f'{name} — Éc.-type'] = round(s.std() * (100 if var in ['underpricing','profitable','ret_30d'] else 1), 2)
        row[f'{name} — N'] = len(s)
    desc_table.append(row)

desc_df = pd.DataFrame(desc_table)
results['desc'] = desc_df
print("\n── Statistiques descriptives ──")
print(desc_df.to_string(index=False))

# ── 2.2 Confidence Intervals 95% (Bootstrap) ────────────────────────────────
def bootstrap_ci(data, stat_fn=np.mean, n_boot=2000, ci=0.95):
    boots = [stat_fn(np.random.choice(data, len(data), replace=True)) for _ in range(n_boot)]
    alpha = (1 - ci) / 2
    return np.percentile(boots, [alpha*100, (1-alpha)*100])

ci_results = {}
for var in ['underpricing', 'revenue_ltm', 'profitable', 'ipo_size_m']:
    ci_results[var] = {
        'low': bootstrap_ci(low[var].dropna().values),
        'high': bootstrap_ci(high[var].dropna().values),
    }
results['ci'] = ci_results

# ── 2.3 Mann-Whitney Tests ───────────────────────────────────────────────────
mw_results = {}

# H1 — Revenue (selectivity)
stat, p = stats.mannwhitneyu(high['revenue_ltm'], low['revenue_ltm'], alternative='greater')
mw_results['H1_revenue'] = {'stat': stat, 'p': p, 'sig': p < 0.05,
                              'h_low': low['revenue_ltm'].median(), 'h_high': high['revenue_ltm'].median()}

# H1 — Profitability
stat, p = stats.mannwhitneyu(high['profitable'], low['profitable'], alternative='greater')
mw_results['H1_profit'] = {'stat': stat, 'p': p, 'sig': p < 0.05,
                             'h_low': low['profitable'].mean(), 'h_high': high['profitable'].mean()}

# H2 — Underpricing
stat, p = stats.mannwhitneyu(low['underpricing'], high['underpricing'], alternative='greater')
mw_results['H2_underpricing'] = {'stat': stat, 'p': p, 'sig': p < 0.05,
                                   'h_low': low['underpricing'].mean(), 'h_high': high['underpricing'].mean()}

# H3 — Volatility at 30d
stat, p = stats.mannwhitneyu(np.abs(low['ret_30d']), np.abs(high['ret_30d']), alternative='greater')
mw_results['H3_volatility'] = {'stat': stat, 'p': p, 'sig': p < 0.05,
                                 'std_low': low['ret_30d'].std(), 'std_high': high['ret_30d'].std()}

# H4 — 12-month performance (2022-2023 only)
h4_low = low_h4['ret_12m'].dropna()
h4_high = high_h4['ret_12m'].dropna()
stat, p = stats.mannwhitneyu(h4_high, h4_low, alternative='greater')
mw_results['H4_12m'] = {'stat': stat, 'p': p, 'sig': p < 0.05,
                         'h_low': h4_low.mean(), 'h_high': h4_high.mean()}

results['mw'] = mw_results

print("\n── Tests de Mann-Whitney ──")
for k, v in mw_results.items():
    sig_str = "✓ Significatif" if v['sig'] else "✗ Non significatif"
    print(f"  {k}: U={v['stat']:.0f}, p={v['p']:.4f}  → {sig_str}")

# ── 2.4 Robustness: excluding 2021 ──────────────────────────────────────────
low_no21 = all_ipo[all_ipo['year'].isin([2019, 2020])]
stat_r, p_r = stats.mannwhitneyu(high['revenue_ltm'], low_no21['revenue_ltm'], alternative='greater')
stat_u, p_u = stats.mannwhitneyu(low_no21['underpricing'], high['underpricing'], alternative='greater')
results['robustness'] = {
    'H1_no2021': {'stat': stat_r, 'p': p_r, 'sig': p_r < 0.05},
    'H2_no2021': {'stat': stat_u, 'p': p_u, 'sig': p_u < 0.05},
}
print(f"\n── Robustesse (sans 2021): H1 p={p_r:.4f}, H2 p={p_u:.4f} ──")

# ── 2.5 OLS Regression ──────────────────────────────────────────────────────
import statsmodels.api as sm

reg_df = all_ipo[['underpricing', 'fed_rate', 'ipo_size_m', 'profitable', 'pe_backed', 'sector']].dropna()
reg_df = reg_df.copy()
reg_df['log_size'] = np.log(reg_df['ipo_size_m'])
reg_df['profitable'] = reg_df['profitable'].astype(float)
reg_df['pe_backed'] = reg_df['pe_backed'].astype(float)

# Sector dummies (Tech as reference)
sector_dummies = pd.get_dummies(reg_df['sector'], prefix='sec', drop_first=False).astype(float)
sector_dummies = sector_dummies.drop(columns=['sec_Tech'], errors='ignore')

X = pd.concat([reg_df[['fed_rate', 'log_size', 'profitable', 'pe_backed']], sector_dummies], axis=1)
X = sm.add_constant(X)
y = reg_df['underpricing']

model = sm.OLS(y, X).fit()
results['ols'] = model

print("\n── Régression OLS ──")
print(f"  R² = {model.rsquared:.4f}, R² ajusté = {model.rsquared_adj:.4f}")
print(f"  Coeff. fed_rate = {model.params['fed_rate']:.4f}, p = {model.pvalues['fed_rate']:.4f}")
print(f"  Coeff. log_size = {model.params['log_size']:.4f}, p = {model.pvalues['log_size']:.4f}")
print(f"  Coeff. profitable = {model.params['profitable']:.4f}, p = {model.pvalues['profitable']:.4f}")
print(f"  Coeff. pe_backed = {model.params['pe_backed']:.4f}, p = {model.pvalues['pe_backed']:.4f}")


# ─────────────────────────────────────────────────────────────────────────────
# PHASE 3 — FIGURES
# ─────────────────────────────────────────────────────────────────────────────

COLORS = {
    'low': '#2C7BB6',    # blue
    'high': '#D7191C',   # red
    'accent': '#1A1A2E',
    'grid': '#E8E8E8',
}

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.4,
    'grid.color': COLORS['grid'],
})

# ── Figure 1: Taux Fed + Volume IPO ─────────────────────────────────────────
fig1, ax1 = plt.subplots(figsize=(11, 5))
years = [2019, 2020, 2021, 2022, 2023, 2024]
ipo_counts = [232, 480, 397, 28, 35, 62]
fed_rates = [2.40, 0.25, 0.08, 3.80, 5.25, 5.40]

ax2 = ax1.twinx()
bars = ax1.bar(years, ipo_counts, color=[COLORS['low']]*3 + [COLORS['high']]*3,
               alpha=0.75, width=0.6, zorder=3)
line, = ax2.plot(years, fed_rates, 'ko-', linewidth=2.5, markersize=7, zorder=4, label='Taux Fed Funds (%)')

ax1.set_xlabel('Année', fontsize=12)
ax1.set_ylabel('Nombre d\'IPO traditionnelles', fontsize=12, color='#333')
ax2.set_ylabel('Taux directeur Fed (%)', fontsize=12, color='#333')
ax1.set_title('Volume des IPO US vs Taux directeur de la Fed (2019–2024)', fontsize=13, fontweight='bold', pad=15)
ax1.set_ylim(0, 580)
ax2.set_ylim(0, 7)

patch_low = mpatches.Patch(color=COLORS['low'], alpha=0.75, label='Cohorte Taux bas (2019–2021)')
patch_high = mpatches.Patch(color=COLORS['high'], alpha=0.75, label='Cohorte Taux élevés (2022–2024)')
ax1.legend(handles=[patch_low, patch_high, line], loc='upper right', fontsize=9)

for bar, count in zip(bars, ipo_counts):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 8, str(count),
             ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.tight_layout()
fig1.savefig('fig1_volume_rates.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fig 1 saved.")

# ── Figure 2: Underpricing Distribution (boxplot + violin) ──────────────────
fig2, axes = plt.subplots(1, 2, figsize=(12, 5))

# Boxplot by cohort
up_data = [low['underpricing']*100, high['underpricing']*100]
bp = axes[0].boxplot(up_data, labels=['Taux bas\n(2019–2021)', 'Taux élevés\n(2022–2024)'],
                      patch_artist=True, widths=0.5,
                      medianprops=dict(color='black', linewidth=2))
bp['boxes'][0].set_facecolor(COLORS['low'])
bp['boxes'][1].set_facecolor(COLORS['high'])
for patch in bp['boxes']:
    patch.set_alpha(0.7)
axes[0].set_title('Distribution de l\'Underpricing J0', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Underpricing (%)', fontsize=10)
axes[0].axhline(y=0, color='gray', linestyle='--', alpha=0.5)

# Yearly trend
yearly_up = all_ipo.groupby('year')['underpricing'].mean() * 100
colors_bars = [COLORS['low']]*3 + [COLORS['high']]*3
axes[1].bar(yearly_up.index, yearly_up.values, color=colors_bars, alpha=0.8, width=0.6)
axes[1].set_title('Underpricing Moyen par Année (%)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Underpricing moyen (%)', fontsize=10)
axes[1].set_xlabel('Année', fontsize=10)
for x, y in zip(yearly_up.index, yearly_up.values):
    axes[1].text(x, y + 0.3, f'{y:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
fig2.savefig('fig2_underpricing.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fig 2 saved.")

# ── Figure 3: Revenue & Profitability Comparison ─────────────────────────────
fig3, axes = plt.subplots(1, 2, figsize=(12, 5))

# Revenue by year
yearly_rev = all_ipo.groupby('year')['revenue_ltm'].median()
colors_bars = [COLORS['low']]*3 + [COLORS['high']]*3
axes[0].bar(yearly_rev.index, yearly_rev.values, color=colors_bars, alpha=0.8, width=0.6)
axes[0].set_title('CA Médian LTM par Année (M$)', fontsize=11, fontweight='bold')
axes[0].set_ylabel('Chiffre d\'affaires médian (M$)', fontsize=10)
axes[0].set_xlabel('Année', fontsize=10)
for x, y in zip(yearly_rev.index, yearly_rev.values):
    axes[0].text(x, y + 5, f'{y:.0f}M$', ha='center', va='bottom', fontsize=8, fontweight='bold')

# Profitability % by year
yearly_prof = all_ipo.groupby('year')['profitable'].mean() * 100
axes[1].bar(yearly_prof.index, yearly_prof.values, color=colors_bars, alpha=0.8, width=0.6)
axes[1].axhline(y=50, color='gray', linestyle='--', alpha=0.6, label='Seuil 50%')
axes[1].set_title('Part d\'Entreprises Rentables par Année (%)', fontsize=11, fontweight='bold')
axes[1].set_ylabel('Part rentable (%)', fontsize=10)
axes[1].set_xlabel('Année', fontsize=10)
axes[1].legend(fontsize=9)
for x, y in zip(yearly_prof.index, yearly_prof.values):
    axes[1].text(x, y + 0.8, f'{y:.1f}%', ha='center', va='bottom', fontsize=8, fontweight='bold')

plt.tight_layout()
fig3.savefig('fig3_fundamentals.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fig 3 saved.")

# ── Figure 4: Confidence Intervals ──────────────────────────────────────────
fig4, axes = plt.subplots(1, 2, figsize=(12, 5))

# Underpricing CI
ci_vars = ['underpricing', 'revenue_ltm']
ci_titles = ['Underpricing J0 (%)', 'CA Médian LTM (M$)']
ci_scales = [100, 1]

for ax, var, title, scale in zip(axes, ci_vars, ci_titles, ci_scales):
    for i, (name, cohort, color) in enumerate([('Taux bas', low, COLORS['low']), ('Taux élevés', high, COLORS['high'])]):
        ci = ci_results[var][('low' if name == 'Taux bas' else 'high')]
        mean_val = cohort[var].mean() * scale
        ci_lo = ci[0] * scale
        ci_hi = ci[1] * scale
        ax.errorbar(i, mean_val, yerr=[[mean_val - ci_lo], [ci_hi - mean_val]],
                    fmt='o', color=color, markersize=10, capsize=8, capthick=2, elinewidth=2)
        ax.text(i, ci_hi + (ci_hi - ci_lo) * 0.1, f'{mean_val:.1f}', ha='center', fontsize=10, fontweight='bold', color=color)
    ax.set_xticks([0, 1])
    ax.set_xticklabels(['Taux bas\n(2019–2021)', 'Taux élevés\n(2022–2024)'], fontsize=10)
    ax.set_title(f'IC 95% — {title}', fontsize=11, fontweight='bold')
    ax.set_ylabel(title, fontsize=10)

plt.tight_layout()
fig4.savefig('fig4_confidence_intervals.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fig 4 saved.")

# ── Figure 5: OLS Coefficients ───────────────────────────────────────────────
fig5, ax = plt.subplots(figsize=(9, 5))
coef_vars = ['fed_rate', 'log_size', 'profitable', 'pe_backed']
coef_labels = ['Taux Fed Funds', 'Log(Taille émission)', 'Rentabilité (dummy)', 'Sponsor PE (dummy)']
coefs = [model.params[v] for v in coef_vars]
pvals = [model.pvalues[v] for v in coef_vars]
errs = [model.bse[v] * 1.96 for v in coef_vars]

colors_coef = [COLORS['high'] if c < 0 else COLORS['low'] for c in coefs]
bars_c = ax.barh(coef_labels, coefs, xerr=errs, color=colors_coef, alpha=0.8,
                  error_kw={'linewidth': 2, 'capsize': 5})
ax.axvline(x=0, color='black', linewidth=1.2)
for i, (c, p, err) in enumerate(zip(coefs, pvals, errs)):
    sig = '***' if p < 0.001 else '**' if p < 0.01 else '*' if p < 0.05 else 'n.s.'
    ax.text(c + (err + 0.003) * np.sign(c), i, sig, va='center', ha='left' if c > 0 else 'right', fontsize=11, fontweight='bold')

ax.set_title(f'Régression OLS — Coefficients (Variable dépendante : Underpricing J0)\nR² = {model.rsquared:.3f}',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Coefficient', fontsize=10)

plt.tight_layout()
fig5.savefig('fig5_ols.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fig 5 saved.")

# ── Figure 6: Sector Distribution ────────────────────────────────────────────
fig6, axes = plt.subplots(1, 2, figsize=(11, 5))
sector_colors = ['#2C7BB6', '#D7191C', '#1A9641', '#FDAE61', '#762A83', '#A6DBA0']

for ax, (name, cohort) in zip(axes, [('Taux bas (2019–2021)', low), ('Taux élevés (2022–2024)', high)]):
    sec_counts = cohort['sector'].value_counts()
    wedges, texts, autotexts = ax.pie(sec_counts.values, labels=sec_counts.index,
                                       autopct='%1.1f%%', colors=sector_colors[:len(sec_counts)],
                                       pctdistance=0.85, startangle=90)
    for at in autotexts:
        at.set_fontsize(9)
    ax.set_title(f'Répartition Sectorielle\n{name}', fontsize=10, fontweight='bold')

plt.tight_layout()
fig6.savefig('fig6_sectors.png', dpi=150, bbox_inches='tight')
plt.close()
print("Fig 6 saved.")

print("\n✓ All figures saved. Analysis complete.")

# Save results dict for use in PDF generation
import pickle
with open('results.pkl', 'wb') as f:
    pickle.dump({
        'all_ipo': all_ipo,
        'low': low, 'high': high,
        'desc_df': desc_df,
        'ci_results': ci_results,
        'mw_results': mw_results,
        'model': model,
        'results': results,
    }, f)
print("✓ Results saved to pickle.")

Total IPOs: 1131
Low-rate cohort: 1009 IPOs ({2019: 218, 2020: 433, 2021: 358})
High-rate cohort: 122 IPOs ({2022: 27, 2023: 34, 2024: 61})

── Statistiques descriptives ──
                   Variable  Taux bas — Moyenne  Taux bas — Médiane  Taux bas — Éc.-type  Taux bas — N  Taux élevés — Moyenne  Taux élevés — Médiane  Taux élevés — Éc.-type  Taux élevés — N
        Underpricing J0 (%)               27.61               24.97                37.87          1009                  15.67                  16.46                   23.04              122
Chiffre d'affaires LTM (M$)              195.86              168.28               144.47          1009                 622.82                 550.05                  430.89              122
          Part rentable (%)               41.03                0.00                49.21          1009                  74.59                 100.00                   43.71              122
       Taille émission (M$)              323.03              286.42